# 04 - Feature Engineering
Build smarter features from cleaned F1 2023 data for ML model training.
- Driver performance features
- Team performance features
- Grid position features
- Save final ML-ready dataset

In [1]:
import pandas as pd
import numpy as np
import os

In [23]:
df = pd.read_csv('../data/f1_2023_cleaned.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (440, 14)
Columns: ['DriverNumber', 'BroadcastName', 'FullName', 'TeamName', 'Position', 'Points', 'Status', 'GridPosition', 'Year', 'RaceName', 'Round', 'Top3', 'Finished', 'TeamID']


,DriverNumber,BroadcastName,FullName,TeamName,Position,Points,Status,GridPosition,Year,RaceName,Round,Top3,Finished,TeamID
0,1,M VERSTAPPEN,Max Verstappen,Red Bull Racing,1,25.0,Finished,1,2023,Bahrain,1,1,1,0
1,11,S PEREZ,Sergio Perez,Red Bull Racing,2,18.0,Finished,2,2023,Bahrain,1,1,1,0
2,14,F ALONSO,Fernando Alonso,Aston Martin,3,15.0,Finished,5,2023,Bahrain,1,1,1,1
3,55,C SAINZ,Carlos Sainz,Ferrari,4,12.0,Finished,4,2023,Bahrain,1,0,1,2
4,44,L HAMILTON,Lewis Hamilton,Mercedes,5,10.0,Finished,7,2023,Bahrain,1,0,1,3


In [24]:
df = df.sort_values(['Round', 'Position']).reset_index(drop=True)
print("Data sorted by round!")
df[['RaceName', 'Round', 'FullName', 'Position']].head(10)

Data sorted by round!


,RaceName,Round,FullName,Position
0,Bahrain,1,Max Verstappen,1
1,Bahrain,1,Sergio Perez,2
2,Bahrain,1,Fernando Alonso,3
3,Bahrain,1,Carlos Sainz,4
4,Bahrain,1,Lewis Hamilton,5
5,Bahrain,1,Lance Stroll,6
6,Bahrain,1,George Russell,7
7,Bahrain,1,Valtteri Bottas,8
8,Bahrain,1,Pierre Gasly,9
9,Bahrain,1,Alexander Albon,10


# Feature 1: Driver's cumulative top3 count

In [27]:
# For each driver, count their top3 finishes in all PREVIOUS races
df['Driver_Top3_SoFar'] = (
    df.groupby('FullName')['Top3']
    .transform(lambda x: x.shift(1).expanding().sum().fillna(0))
)

print("Driver_Top3_SoFar created!\n")
# Check Verstappen's progression
print("do do do do max verstappen\n")
vmax = df[df['FullName'].str.contains('Verstappen')]
print(vmax[['RaceName', 'Round', 'Position', 'Top3', 'Driver_Top3_SoFar']].head(25),"\n")

# Check leclerc's progression
print("charles leclerc\n")
charles = df[df['FullName'].str.contains('Charles Leclerc')]
print(charles[['RaceName', 'Round', 'Position', 'Top3', 'Driver_Top3_SoFar']].head(25))

Driver_Top3_SoFar created!

do do do do max verstappen

          RaceName  Round  Position  Top3  Driver_Top3_SoFar
0          Bahrain      1         1     1                0.0
21    Saudi Arabia      2         2     1                1.0
40       Australia      3         1     1                2.0
61      Azerbaijan      4         2     1                3.0
80           Miami      5         1     1                4.0
100         Monaco      6         1     1                5.0
120          Spain      7         1     1                6.0
140         Canada      8         1     1                7.0
160        Austria      9         1     1                8.0
180        Britain     10         1     1                9.0
200        Hungary     11         1     1               10.0
220        Belgium     12         1     1               11.0
240    Netherlands     13         1     1               12.0
260          Italy     14         1     1               13.0
284      Singapore     15    

# Feature 2: Driver's average finish position (last 3 races)

In [26]:
df['Driver_AvgPos_Last3'] = (
    df.groupby('FullName')['Position']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean().fillna(10))
)

print("Driver_AvgPos_Last3 created!")
vmax = df[df['FullName'].str.contains('Verstappen')]
print(vmax[['RaceName', 'Round', 'Position', 'Driver_AvgPos_Last3']].head(10))

Driver_AvgPos_Last3 created!
         RaceName  Round  Position  Driver_AvgPos_Last3
0         Bahrain      1         1            10.000000
21   Saudi Arabia      2         2             1.000000
40      Australia      3         1             1.500000
61     Azerbaijan      4         2             1.333333
80          Miami      5         1             1.666667
100        Monaco      6         1             1.333333
120         Spain      7         1             1.333333
140        Canada      8         1             1.000000
160       Austria      9         1             1.000000
180       Britain     10         1             1.000000


# Feature 3: Driver's finish rate (how often they finish vs DNF)

In [33]:
df['Driver_FinishRate'] = (
    df.groupby('FullName')['Finished']
    .transform(lambda x: x.shift(1).expanding().mean().fillna(0.8))
)

print("Driver_FinishRate created!")
print(df.groupby('FullName')['Driver_FinishRate'].last().sort_values())

Driver_FinishRate created!
FullName
Nyck De Vries       0.222222
Kevin Magnussen     0.333333
Logan Sargeant      0.380952
Nico Hulkenberg     0.428571
Daniel Ricciardo    0.500000
Guanyu Zhou         0.523810
Esteban Ocon        0.571429
Valtteri Bottas     0.571429
Yuki Tsunoda        0.619048
Oscar Piastri       0.619048
Alexander Albon     0.666667
Lance Stroll        0.714286
Liam Lawson         0.750000
Charles Leclerc     0.761905
George Russell      0.809524
Lando Norris        0.809524
Pierre Gasly        0.857143
Sergio Perez        0.857143
Lewis Hamilton      0.904762
Fernando Alonso     0.904762
Carlos Sainz        0.904762
Max Verstappen      1.000000
Name: Driver_FinishRate, dtype: float64


# Feature 4: Team's cumulative top3 count

In [34]:
df['Team_Top3_SoFar'] = (
    df.groupby('TeamName')['Top3']
    .transform(lambda x: x.shift(1).expanding().sum().fillna(0))
)

print("Team_Top3_SoFar created!")
print(df.groupby('TeamName')['Team_Top3_SoFar'].last().sort_values(ascending=False))

Team_Top3_SoFar created!
TeamName
Red Bull Racing    30.0
Ferrari             9.0
McLaren             9.0
Aston Martin        8.0
Mercedes            8.0
Alpine              2.0
Alfa Romeo          0.0
AlphaTauri          0.0
Haas F1 Team        0.0
Williams            0.0
Name: Team_Top3_SoFar, dtype: float64


# Feature 5: Team's average finish position

In [35]:
df['Team_AvgPos'] = (
    df.groupby('TeamName')['Position']
    .transform(lambda x: x.shift(1).expanding().mean().fillna(10))
)

print("Team_AvgPos created!")
print(df.groupby('TeamName')['Team_AvgPos'].last().sort_values())

Team_AvgPos created!
TeamName
Red Bull Racing     3.558140
Mercedes            7.093023
Ferrari             7.627907
Aston Martin        8.883721
McLaren             9.534884
Alpine             11.232558
AlphaTauri         13.674419
Alfa Romeo         13.883721
Williams           14.069767
Haas F1 Team       14.953488
Name: Team_AvgPos, dtype: float64


# Feature 6: Grid position advantage

In [36]:
df['Grid_vs_Avg'] = (
    df.groupby('Round')['GridPosition']
    .transform(lambda x: x - x.mean())
)

print("Grid_vs_Avg created!")
print(df[['FullName', 'GridPosition', 'Grid_vs_Avg']].head(10))

Grid_vs_Avg created!
          FullName  GridPosition  Grid_vs_Avg
0   Max Verstappen             1         -9.5
1     Sergio Perez             2         -8.5
2  Fernando Alonso             5         -5.5
3     Carlos Sainz             4         -6.5
4   Lewis Hamilton             7         -3.5
5     Lance Stroll             8         -2.5
6   George Russell             6         -4.5
7  Valtteri Bottas            12          1.5
8     Pierre Gasly            20          9.5
9  Alexander Albon            15          4.5


# Feature 7: Is this a top team?

In [37]:
top_teams = ['Red Bull Racing', 'Mercedes', 'Ferrari', 'McLaren']

df['IsTopTeam'] = df['TeamName'].apply(
    lambda x: 1 if x in top_teams else 0
)

print("IsTopTeam created!")
print(df['IsTopTeam'].value_counts())

IsTopTeam created!
IsTopTeam
0    264
1    176
Name: count, dtype: int64


# Feature 8: GridPosition squared

In [38]:
df['GridPosition_Squared'] = df['GridPosition'] ** 2

print("GridPosition_Squared created!")
print(df[['GridPosition', 'GridPosition_Squared']].head(5))

GridPosition_Squared created!
   GridPosition  GridPosition_Squared
0             1                     1
1             2                     4
2             5                    25
3             4                    16
4             7                    49


# Check all new features

In [39]:
print("All columns now:")
print(df.columns.tolist())
print("\nShape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

All columns now:
['DriverNumber', 'BroadcastName', 'FullName', 'TeamName', 'Position', 'Points', 'Status', 'GridPosition', 'Year', 'RaceName', 'Round', 'Top3', 'Finished', 'TeamID', 'Driver_Top3_SoFar', 'Driver_AvgPos_Last3', 'Driver_FinishRate', 'Team_Top3_SoFar', 'Team_AvgPos', 'Grid_vs_Avg', 'IsTopTeam', 'GridPosition_Squared']

Shape: (440, 22)

Missing values:
DriverNumber            0
BroadcastName           1
FullName                0
TeamName                0
Position                0
Points                  0
Status                  0
GridPosition            0
Year                    0
RaceName                0
Round                   0
Top3                    0
Finished                0
TeamID                  0
Driver_Top3_SoFar       0
Driver_AvgPos_Last3     0
Driver_FinishRate       0
Team_Top3_SoFar         0
Team_AvgPos             0
Grid_vs_Avg             0
IsTopTeam               0
GridPosition_Squared    0
dtype: int64


# Select final ML features

In [40]:
feature_columns = [
    'GridPosition',
    'GridPosition_Squared',
    'Grid_vs_Avg',
    'TeamID',
    'IsTopTeam',
    'Points',
    'Finished',
    'Driver_Top3_SoFar',
    'Driver_AvgPos_Last3',
    'Driver_FinishRate',
    'Team_Top3_SoFar',
    'Team_AvgPos',
    'Top3'  # this is what we predict
]

df_ml = df[feature_columns].copy()

print("ML dataset shape:", df_ml.shape)
print("\nFirst few rows:")
df_ml.head()

ML dataset shape: (440, 13)

First few rows:


,GridPosition,GridPosition_Squared,Grid_vs_Avg,TeamID,IsTopTeam,Points,Finished,Driver_Top3_SoFar,Driver_AvgPos_Last3,Driver_FinishRate,Team_Top3_SoFar,Team_AvgPos,Top3
0,1,1,-9.5,0,1,25.0,1,0.0,10.0,0.8,0.0,10.0,1
1,2,4,-8.5,0,1,18.0,1,0.0,10.0,0.8,1.0,1.0,1
2,5,25,-5.5,1,0,15.0,1,0.0,10.0,0.8,0.0,10.0,1
3,4,16,-6.5,2,1,12.0,1,0.0,10.0,0.8,0.0,10.0,0
4,7,49,-3.5,3,1,10.0,1,0.0,10.0,0.8,0.0,10.0,0


# Check Top3 balance

In [41]:
print("Top3 distribution:")
print(df_ml['Top3'].value_counts())
print(f"\nTop3 rate: {df_ml['Top3'].mean():.1%}")


Top3 distribution:
Top3
0    374
1     66
Name: count, dtype: int64

Top3 rate: 15.0%


# Check feature correlations with Top3

In [42]:
correlations = df_ml.corr()['Top3'].sort_values(ascending=False)
print("Feature correlations with Top3:")
print(correlations)

Feature correlations with Top3:
Top3                    1.000000
Points                  0.836348
Driver_Top3_SoFar       0.459843
Team_Top3_SoFar         0.392070
IsTopTeam               0.384573
Driver_FinishRate       0.356870
Finished                0.280976
TeamID                 -0.345692
GridPosition_Squared   -0.393850
Driver_AvgPos_Last3    -0.467085
Team_AvgPos            -0.477762
GridPosition           -0.487281
Grid_vs_Avg            -0.487886
Name: Top3, dtype: float64


# Save the ML ready dataset

In [43]:
os.makedirs('../data', exist_ok=True)
df_ml.to_csv('../data/f1_2023_ml_ready.csv', index=False)
print(f"ML ready dataset saved! Shape: {df_ml.shape} ✅")

ML ready dataset saved! Shape: (440, 13) ✅
